In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA04 - Gifts/Travel or Lodging Expenses_Non POs
   
   BUSINESS QUESTION: 
   During the assessment period, how many instances were recorded by the Unit where 
   employees provide or receive gifts or paid for travel/entertainment to Non POs 
   exceeding the $250 threshold?
   
   LOGIC & ARCHITECTURE SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
   2. COUPA DATA ONLY: Uses the combined Coupa file, keeping only rows that
      exist in the original 7 files (removes new rows via NOT EXISTS).
   3. ACCOUNT PARSING: Splits the Account string to extract the Cost Center (index 0) 
      and the Category Code (index 2).
   4. THRESHOLD RULE: Normalizes all amount-like tokens found in `Total`, uses the
      maximum parsed amount, divides by the Divisor (per-person amount), then
      enforces Per_Person_Total > 250.
   5. NON-PUBLIC OFFICIAL FILTER: Strictly filters for 'N' or 'NO'.
   6. TARGET CATEGORY CODES: Hardcoded list of 29 specific category codes.
   7. THE 793 EXCEPTION RULE: If the Category Code is '793', it is IGNORED unless 
      the mapped AU is strictly '101016' (TD Asset Management).
   8. FINAL OUTPUT: Returns findings divided by non-PO mapped source records in the AU, 
      anchored to the Master List and formatted as a percentage.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Old_Coupa_Union AS (
    -- 2a. Union the original 7 Coupa files for row-level comparison
    SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

New_Coupa_Rows AS (
    -- 2b. Identify newly added rows in the combined table (not in old files).
    --     Expensecategory excluded from comparison (format differs between old/new).
    SELECT c.DateExpenseIncurrred, c.PublicOfficial, c.Total, c.BusinessPurpose, c.Account
    FROM hive_metastore.ra_adido_2025.ritm16070584_combined_04272026 c
    WHERE NOT EXISTS (
        SELECT 1 FROM Old_Coupa_Union o
        WHERE TRIM(c.DateExpenseIncurrred) <=> TRIM(o.DateExpenseIncurrred)
          AND TRIM(c.PublicOfficial) <=> TRIM(o.PublicOfficial)
          AND TRIM(c.Total) <=> TRIM(o.Total)
          AND TRIM(c.BusinessPurpose) <=> TRIM(o.BusinessPurpose)
          AND TRIM(c.Account) <=> TRIM(o.Account)
    )
),

Combined_Coupa_Raw AS (
    -- 2c. Reduced combined table: remove newly added rows, keep Divisor.
    SELECT c.Account, c.PublicOfficial, c.Total, c.Divisor
    FROM hive_metastore.ra_adido_2025.ritm16070584_combined_04272026 c
    WHERE NOT EXISTS (
        SELECT 1 FROM New_Coupa_Rows n
        WHERE TRIM(c.DateExpenseIncurrred) <=> TRIM(n.DateExpenseIncurrred)
          AND TRIM(c.PublicOfficial) <=> TRIM(n.PublicOfficial)
          AND TRIM(c.Total) <=> TRIM(n.Total)
          AND TRIM(c.BusinessPurpose) <=> TRIM(n.BusinessPurpose)
          AND TRIM(c.Account) <=> TRIM(n.Account)
    )
),

Coupa_Parsed AS (
    -- 3 & 4. Parse the Account string and clean the Total for mathematical comparison
    SELECT 
        CASE WHEN TRIM(SPLIT(Account, '-')[0]) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(SPLIT(Account, '-')[0]), 4), 4, '0') ELSE UPPER(TRIM(SPLIT(Account, '-')[0])) END AS Cleaned_CC,
        TRIM(SPLIT(Account, '-')[2]) AS CatCode,
        TRIM(PublicOfficial) AS PublicOfficial,
        aggregate(
            filter(
                transform(
                    filter(
                        split(TRIM(REGEXP_REPLACE(REGEXP_REPLACE(COALESCE(Total, ''), '[^0-9,\\.]', ' '), '\\s+', ' ')), ' '),
                        token -> token RLIKE '.*[0-9].*'
                    ),
                    token -> TRY_CAST(
                        CASE
                            WHEN token RLIKE '^[0-9]+,[0-9]{2}$' THEN REPLACE(token, ',', '.')
                            WHEN token RLIKE '^[0-9]{1,3}(\\.[0-9]{3})+,[0-9]{2}$' THEN REPLACE(REPLACE(token, '.', ''), ',', '.')
                            WHEN token RLIKE '^[0-9]{1,3}(,[0-9]{3})+\\.[0-9]{2}$' THEN REPLACE(token, ',', '')
                            WHEN token RLIKE '^[0-9]+(\\.[0-9]{2})?$' THEN token
                            ELSE NULL
                        END AS DECIMAL(10,2)
                    )
                ),
                amount -> amount IS NOT NULL
            ),
            CAST(NULL AS DECIMAL(10,2)),
            (acc, amount) -> CASE WHEN acc IS NULL OR amount > acc THEN amount ELSE acc END
        ) AS Numeric_Total,
        GREATEST(COALESCE(TRY_CAST(REGEXP_REPLACE(Divisor, '[^0-9.]', '') AS DECIMAL(10,2)), 1), 1) AS Numeric_Divisor
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
),

CC_Mapping AS (
    -- Standardize the CC Mapping table from our bootstrap
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Flagged_Transactions AS (
    -- 5, 6, 7. Apply all Business Rules to filter valid transactions
    SELECT 
        c.Cleaned_CC,
        c.CatCode,
        m.AU_ID
    FROM Coupa_Parsed c
    INNER JOIN CC_Mapping m
        ON c.Cleaned_CC = m.Mapped_CC
    WHERE 
        -- RULE: Must NOT be a Public Official
        UPPER(TRIM(c.PublicOfficial)) IN ('N', 'NO')
        
        -- RULE: Per-person threshold must be strictly greater than 250
        AND (c.Numeric_Total / c.Numeric_Divisor) > 250
        
        -- RULE: Must be one of the target category codes
        AND c.CatCode IN (
            '066', '009', '012', '067', '095', '134', '168', '192', '203', '204', 
            '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', 
            '639', '676', '782', '783', '784', '792', '793', '890', '892'
        )
        
        -- RULE: The 793 Exception Rule (Ignores 793 unless the mapped AU is 101016)
        AND (c.CatCode != '793' OR (c.CatCode = '793' AND m.AU_ID = '101016'))
),

Flagged_AUs AS (
    -- Count qualifying findings per AU for the numerator
    SELECT 
        AU_ID,
        COUNT(*) AS Flagged_Count
    FROM Flagged_Transactions 
    GROUP BY AU_ID
),

Total_Records_By_AU AS (
    -- Count non-PO mapped source records per AU for the denominator
    SELECT 
        m.AU_ID,
        COUNT(*) AS Total_Record_Count
    FROM Coupa_Parsed c
    INNER JOIN CC_Mapping m
        ON c.Cleaned_CC = m.Mapped_CC
    WHERE UPPER(TRIM(c.PublicOfficial)) IN ('N', 'NO')
    GROUP BY m.AU_ID
)

-- 8. Final Output: Strictly structured to 6 columns, anchored to the Master list
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'EBA04' AS QuestionID,               
    CASE 
        WHEN COALESCE(r.Total_Record_Count, 0) = 0 THEN '0%'
        ELSE CAST(ROUND((CAST(COALESCE(f.Flagged_Count, 0) AS DOUBLE) / r.Total_Record_Count) * 100, 2) AS STRING) || '%'
    END AS Response,
    a.In_ABAC_AU_List
FROM Master_AUs a
LEFT JOIN Flagged_AUs f
    ON a.BusinessID = f.AU_ID
LEFT JOIN Total_Records_By_AU r
    ON a.BusinessID = r.AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA04 - AU Level Calculation Review
   PURPOSE: One row per AU showing normalized Cost Centers and how the final record-
   based percentage was calculated.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Old_Coupa_Union AS (
    -- 2a. Union the original 7 Coupa files for row-level comparison
    SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

New_Coupa_Rows AS (
    -- 2b. Identify newly added rows in the combined table (not in old files).
    --     Expensecategory excluded from comparison (format differs between old/new).
    SELECT c.DateExpenseIncurrred, c.PublicOfficial, c.Total, c.BusinessPurpose, c.Account
    FROM hive_metastore.ra_adido_2025.ritm16070584_combined_04272026 c
    WHERE NOT EXISTS (
        SELECT 1 FROM Old_Coupa_Union o
        WHERE TRIM(c.DateExpenseIncurrred) <=> TRIM(o.DateExpenseIncurrred)
          AND TRIM(c.PublicOfficial) <=> TRIM(o.PublicOfficial)
          AND TRIM(c.Total) <=> TRIM(o.Total)
          AND TRIM(c.BusinessPurpose) <=> TRIM(o.BusinessPurpose)
          AND TRIM(c.Account) <=> TRIM(o.Account)
    )
),

Combined_Coupa_Raw AS (
    -- 2c. Reduced combined table: remove newly added rows, keep Divisor.
    SELECT c.Account, c.PublicOfficial, c.Total, c.Divisor
    FROM hive_metastore.ra_adido_2025.ritm16070584_combined_04272026 c
    WHERE NOT EXISTS (
        SELECT 1 FROM New_Coupa_Rows n
        WHERE TRIM(c.DateExpenseIncurrred) <=> TRIM(n.DateExpenseIncurrred)
          AND TRIM(c.PublicOfficial) <=> TRIM(n.PublicOfficial)
          AND TRIM(c.Total) <=> TRIM(n.Total)
          AND TRIM(c.BusinessPurpose) <=> TRIM(n.BusinessPurpose)
          AND TRIM(c.Account) <=> TRIM(n.Account)
    )
),

Coupa_Debug_Base AS (
    SELECT 
        Account AS Raw_Account,
        CASE 
            WHEN Account LIKE '%-%-%' THEN 'Pass'
            ELSE 'FAIL: Account not in expected hyphenated format'
        END AS Account_Format_Check,
        TRIM(SPLIT(Account, '-')[0]) AS Raw_Account_CC_Block,
        CASE 
            WHEN Account LIKE '%-%-%' THEN CASE WHEN TRIM(SPLIT(Account, '-')[0]) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(SPLIT(Account, '-')[0]), 4), 4, '0') ELSE UPPER(TRIM(SPLIT(Account, '-')[0])) END
            ELSE NULL
        END AS Cleaned_CC,
        CASE 
            WHEN Account LIKE '%-%-%' THEN TRIM(SPLIT(Account, '-')[2])
            ELSE NULL
        END AS CatCode,
        TRIM(PublicOfficial) AS PublicOfficial,
        Total AS Raw_Total_String,
        aggregate(
            filter(
                transform(
                    filter(
                        split(TRIM(REGEXP_REPLACE(REGEXP_REPLACE(COALESCE(Total, ''), '[^0-9,\\.]', ' '), '\\s+', ' ')), ' '),
                        token -> token RLIKE '.*[0-9].*'
                    ),
                    token -> TRY_CAST(
                        CASE
                            WHEN token RLIKE '^[0-9]+,[0-9]{2}$' THEN REPLACE(token, ',', '.')
                            WHEN token RLIKE '^[0-9]{1,3}(\\.[0-9]{3})+,[0-9]{2}$' THEN REPLACE(REPLACE(token, '.', ''), ',', '.')
                            WHEN token RLIKE '^[0-9]{1,3}(,[0-9]{3})+\\.[0-9]{2}$' THEN REPLACE(token, ',', '')
                            WHEN token RLIKE '^[0-9]+(\\.[0-9]{2})?$' THEN token
                            ELSE NULL
                        END AS DECIMAL(10,2)
                    )
                ),
                amount -> amount IS NOT NULL
            ),
            CAST(NULL AS DECIMAL(10,2)),
            (acc, amount) -> CASE WHEN acc IS NULL OR amount > acc THEN amount ELSE acc END
        ) AS Numeric_Total,
        Divisor AS Raw_Divisor,
        GREATEST(COALESCE(TRY_CAST(REGEXP_REPLACE(Divisor, '[^0-9.]', '') AS DECIMAL(10,2)), 1), 1) AS Numeric_Divisor
    FROM Combined_Coupa_Raw
),

CC_Mapping AS (
    SELECT DISTINCT
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Evaluated_Rows AS (
    SELECT 
        c.Cleaned_CC,
        c.CatCode,
        c.PublicOfficial,
        c.Numeric_Total,
        c.Numeric_Divisor,
        ROUND(c.Numeric_Total / c.Numeric_Divisor, 2) AS Per_Person_Total,
        m.AU_ID AS Mapped_AU_ID,
        mast.BusinessID AS Master_AU_ID,
        mast.AU_Name,
        mast.Business_Segment,
        mast.In_ABAC_AU_List,
        CASE WHEN c.Account_Format_Check = 'Pass' THEN 1 ELSE 0 END AS Parse_Pass,
        CASE WHEN m.AU_ID IS NOT NULL AND mast.BusinessID IS NOT NULL THEN 1 ELSE 0 END AS Bridge_Pass,
        CASE WHEN UPPER(TRIM(c.PublicOfficial)) IN ('N', 'NO') THEN 1 ELSE 0 END AS Non_PO_Pass,
        CASE WHEN c.Numeric_Total IS NOT NULL AND (c.Numeric_Total / c.Numeric_Divisor) > 250 THEN 1 ELSE 0 END AS Threshold_Pass,
        CASE WHEN c.CatCode IN ('066', '009', '012', '067', '095', '134', '168', '192', '203', '204', '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', '639', '676', '782', '783', '784', '792', '793', '890', '892') THEN 1 ELSE 0 END AS CatCode_Pass,
        CASE WHEN c.CatCode = '793' AND mast.BusinessID != '101016' THEN 0 ELSE 1 END AS Exception_793_Pass,
        CASE 
            WHEN c.Account_Format_Check = 'Pass'
             AND mast.BusinessID IS NOT NULL
             AND UPPER(TRIM(c.PublicOfficial)) IN ('N', 'NO')
             AND c.Numeric_Total IS NOT NULL
             AND (c.Numeric_Total / c.Numeric_Divisor) > 250
             AND c.CatCode IN ('066', '009', '012', '067', '095', '134', '168', '192', '203', '204', '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', '639', '676', '782', '783', '784', '792', '793', '890', '892')
             AND NOT (c.CatCode = '793' AND mast.BusinessID != '101016')
            THEN 1 ELSE 0
        END AS Counted_Flag
    FROM Coupa_Debug_Base c
    LEFT JOIN CC_Mapping m
        ON c.Cleaned_CC = m.Mapped_CC
    LEFT JOIN Master_AUs mast
        ON m.AU_ID = mast.BusinessID
),

AU_Debug AS (
    SELECT 
        Master_AU_ID AS AU_ID,
        MAX(AU_Name) AS AU_Name,
        MAX(Business_Segment) AS Business_Segment,
        MAX(In_ABAC_AU_List) AS In_ABAC_AU_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Cleaned_CC))) AS Normalized_CC_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(CASE WHEN Counted_Flag = 1 THEN Cleaned_CC END))) AS Counted_CC_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(CASE WHEN Counted_Flag = 1 THEN CatCode END))) AS Counted_CatCode_List,
        COUNT(*) AS Mapped_Row_Count,
        SUM(Non_PO_Pass) AS Non_PO_Row_Count,
        SUM(Threshold_Pass) AS Above_Threshold_Row_Count,
        SUM(CatCode_Pass) AS Valid_CatCode_Row_Count,
        SUM(Exception_793_Pass) AS Exception_793_Pass_Row_Count,
        SUM(Counted_Flag) AS Counted_Row_Count,
        ROUND(SUM(CASE WHEN Counted_Flag = 1 THEN Numeric_Total END), 2) AS Total_Flagged_Amount,
        ROUND(SUM(CASE WHEN Counted_Flag = 1 THEN Per_Person_Total END), 2) AS Total_Per_Person_Amount,
        ROUND(AVG(CASE WHEN Counted_Flag = 1 THEN Numeric_Divisor END), 2) AS Avg_Divisor,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_LIST(CASE WHEN Counted_Flag = 1 THEN CAST(Numeric_Total AS STRING) END))) AS Flagged_Totals,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_LIST(CASE WHEN Counted_Flag = 1 THEN CAST(Numeric_Divisor AS STRING) END))) AS Flagged_Divisors,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_LIST(CASE WHEN Counted_Flag = 1 THEN CAST(Per_Person_Total AS STRING) END))) AS Flagged_Per_Person
    FROM Evaluated_Rows
    WHERE Master_AU_ID IS NOT NULL
    GROUP BY Master_AU_ID
)

-- Output columns:
-- BusinessID: Master AU ID from the ABAC AU anchor list.
-- AU_Name: Master AU name tied to BusinessID.
-- Business_Segment: Business segment carried from the master AU list.
-- QuestionID: Questionnaire code for this debug output.
-- Response: Final EBA04 percentage shown to the customer.
-- Normalized_CC_List: Normalized cost centers mapped to this AU.
-- Counted_CC_List: Cost centers present on rows included in the numerator.
-- Counted_CatCode_List: Category codes present on rows included in the numerator.
-- Mapped_Row_Count: Full mapped population for this AU before the numerator filters.
-- Non_PO_Row_Count: Rows remaining after removing Public Official records.
-- Above_Threshold_Row_Count: Rows with amount above the EBA04 threshold.
-- Valid_CatCode_Row_Count: Rows that also passed the target category-code filter.
-- Exception_793_Pass_Row_Count: Rows that survived the special 793 exception logic.
-- Counted_Row_Count: Final numerator row count.
-- Total_Flagged_Amount: Sum of Numeric_Total across all counted (flagged) rows.
-- Total_Per_Person_Amount: Sum of per-person amounts (Total/Divisor) across counted rows.
-- Avg_Divisor: Average Divisor value across counted rows (1 means no splitting).
-- Flagged_Totals: Sorted list of raw Total amounts for each counted transaction.
-- Flagged_Divisors: Sorted list of Divisor values for each counted transaction.
-- Flagged_Per_Person: Sorted list of per-person amounts (Total/Divisor) for each counted transaction.
-- Numerator: Duplicate of Counted_Row_Count used in the percentage formula.
-- Denominator: Non-PO row count used as the percentage denominator.
-- Calculated_Percentage: Explicit numerator/denominator percentage before formatting.
-- Calculation_Formula: Plain-English explanation of how Response was produced.
-- In_ABAC_AU_List: Confirms the AU came from the standard ABAC AU list.
SELECT 
    mast.BusinessID,
    mast.AU_Name,
    mast.Business_Segment,
    'EBA04' AS QuestionID,
    CASE 
        WHEN COALESCE(d.Non_PO_Row_Count, 0) = 0 THEN '0%'
        ELSE CAST(ROUND((CAST(COALESCE(d.Counted_Row_Count, 0) AS DOUBLE) / d.Non_PO_Row_Count) * 100, 2) AS STRING) || '%'
    END AS Response,
    COALESCE(d.Normalized_CC_List, 'n/a') AS Normalized_CC_List,
    COALESCE(d.Counted_CC_List, 'n/a') AS Counted_CC_List,
    COALESCE(d.Counted_CatCode_List, 'n/a') AS Counted_CatCode_List,
    COALESCE(d.Mapped_Row_Count, 0) AS Mapped_Row_Count,
    COALESCE(d.Non_PO_Row_Count, 0) AS Non_PO_Row_Count,
    COALESCE(d.Above_Threshold_Row_Count, 0) AS Above_Threshold_Row_Count,
    COALESCE(d.Valid_CatCode_Row_Count, 0) AS Valid_CatCode_Row_Count,
    COALESCE(d.Exception_793_Pass_Row_Count, 0) AS Exception_793_Pass_Row_Count,
    COALESCE(d.Counted_Row_Count, 0) AS Counted_Row_Count,
    COALESCE(d.Total_Flagged_Amount, 0) AS Total_Flagged_Amount,
    COALESCE(d.Total_Per_Person_Amount, 0) AS Total_Per_Person_Amount,
    COALESCE(d.Avg_Divisor, 0) AS Avg_Divisor,
    COALESCE(d.Flagged_Totals, 'n/a') AS Flagged_Totals,
    COALESCE(d.Flagged_Divisors, 'n/a') AS Flagged_Divisors,
    COALESCE(d.Flagged_Per_Person, 'n/a') AS Flagged_Per_Person,
    COALESCE(d.Counted_Row_Count, 0) AS Numerator,
    COALESCE(d.Non_PO_Row_Count, 0) AS Denominator,
    CASE 
        WHEN COALESCE(d.Non_PO_Row_Count, 0) = 0 THEN '0%'
        ELSE CAST(ROUND((CAST(COALESCE(d.Counted_Row_Count, 0) AS DOUBLE) / d.Non_PO_Row_Count) * 100, 2) AS STRING) || '%'
    END AS Calculated_Percentage,
    CASE 
        WHEN COALESCE(d.Non_PO_Row_Count, 0) = 0 THEN CONCAT('Response = 0% because non-PO denominator = 0. Findings = ', CAST(COALESCE(d.Counted_Row_Count, 0) AS STRING), ' and this AU has no non-PO mapped source records.')
        ELSE CONCAT(
            'Response = ', CAST(ROUND((CAST(COALESCE(d.Counted_Row_Count, 0) AS DOUBLE) / d.Non_PO_Row_Count) * 100, 2) AS STRING),
            '% because findings ', CAST(COALESCE(d.Counted_Row_Count, 0) AS STRING), ' / non-PO records ', CAST(d.Non_PO_Row_Count AS STRING),
            ' * 100. Non-PO=', CAST(COALESCE(d.Non_PO_Row_Count, 0) AS STRING),
            ', >250=', CAST(COALESCE(d.Above_Threshold_Row_Count, 0) AS STRING), ', valid CatCode=', CAST(COALESCE(d.Valid_CatCode_Row_Count, 0) AS STRING),
            ', 793-pass=', CAST(COALESCE(d.Exception_793_Pass_Row_Count, 0) AS STRING)
        )
    END AS Calculation_Formula,
    mast.In_ABAC_AU_List
FROM Master_AUs mast
LEFT JOIN AU_Debug d
    ON mast.BusinessID = d.AU_ID
ORDER BY mast.BusinessID;

In [ ]:
# ===================================================================================
# EXPORT: EBA04 AU-level debug table -> eba04_reasons.csv
# Run this cell immediately after the AU-level debug SQL cell above.
# Uses pandas for reliable CSV export (handles newlines, quotes, commas in data).
# ===================================================================================
import re

def safe_rm(path):
    try:
        dbutils.fs.rm(path, recurse=True)
    except Exception:
        pass

# Convert to pandas for reliable CSV writing
pdf = _sqldf.toPandas()

# Clean all string columns: replace newlines, carriage returns, tabs
for col_name in pdf.columns:
    if pdf[col_name].dtype == object:
        pdf[col_name] = pdf[col_name].astype(str).str.replace(r"[\r\n\t]+", " ", regex=True)

# Write to local tmp, then copy to FileStore
local_path = "/tmp/eba04_reasons.csv"
pdf.to_csv(local_path, index=False, encoding="utf-8")

fs_path = "dbfs:/FileStore/eba04_exports/eba04_reasons.csv"
safe_rm(fs_path)
dbutils.fs.mkdirs("dbfs:/FileStore/eba04_exports")
dbutils.fs.cp(f"file:{local_path}", fs_path)

host = spark.conf.get("spark.databricks.workspaceUrl", "<your-databricks-instance>")
print(f"Rows: {len(pdf)}")
print(f"Download: https://{host}/files/eba04_exports/eba04_reasons.csv")


In [ ]:
%sql
/* ===================================================================================
   ROW-LEVEL DEBUG TABLE: EBA04 - Per-Transaction Detail
   PURPOSE: One row per Coupa transaction showing all parsed fields, filter
   evaluations, Divisor impact, and whether the row was counted.
=================================================================================== */

WITH Old_Coupa_Union AS (
    SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

New_Coupa_Rows AS (
    SELECT c.DateExpenseIncurrred, c.PublicOfficial, c.Total, c.BusinessPurpose, c.Account
    FROM hive_metastore.ra_adido_2025.ritm16070584_combined_04272026 c
    WHERE NOT EXISTS (
        SELECT 1 FROM Old_Coupa_Union o
        WHERE TRIM(c.DateExpenseIncurrred) <=> TRIM(o.DateExpenseIncurrred)
          AND TRIM(c.PublicOfficial) <=> TRIM(o.PublicOfficial)
          AND TRIM(c.Total) <=> TRIM(o.Total)
          AND TRIM(c.BusinessPurpose) <=> TRIM(o.BusinessPurpose)
          AND TRIM(c.Account) <=> TRIM(o.Account)
    )
),

Combined_Coupa_Raw AS (
    SELECT c.Account, c.PublicOfficial, c.Total, c.Divisor
    FROM hive_metastore.ra_adido_2025.ritm16070584_combined_04272026 c
    WHERE NOT EXISTS (
        SELECT 1 FROM New_Coupa_Rows n
        WHERE TRIM(c.DateExpenseIncurrred) <=> TRIM(n.DateExpenseIncurrred)
          AND TRIM(c.PublicOfficial) <=> TRIM(n.PublicOfficial)
          AND TRIM(c.Total) <=> TRIM(n.Total)
          AND TRIM(c.BusinessPurpose) <=> TRIM(n.BusinessPurpose)
          AND TRIM(c.Account) <=> TRIM(n.Account)
    )
),

CC_Mapping AS (
    SELECT DISTINCT
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Parsed_Rows AS (
    SELECT
        Account,
        CASE WHEN Account LIKE '%-%-%' THEN CASE WHEN TRIM(SPLIT(Account, '-')[0]) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(SPLIT(Account, '-')[0]), 4), 4, '0') ELSE UPPER(TRIM(SPLIT(Account, '-')[0])) END ELSE NULL END AS Cleaned_CC,
        CASE WHEN Account LIKE '%-%-%' THEN TRIM(SPLIT(Account, '-')[2]) ELSE NULL END AS CatCode,
        TRIM(PublicOfficial) AS PublicOfficial,
        Total AS Raw_Total,
        Divisor AS Raw_Divisor,
        aggregate(
            filter(
                transform(
                    filter(
                        split(TRIM(REGEXP_REPLACE(REGEXP_REPLACE(COALESCE(Total, ''), '[^0-9,\\.]', ' '), '\\s+', ' ')), ' '),
                        token -> token RLIKE '.*[0-9].*'
                    ),
                    token -> TRY_CAST(
                        CASE
                            WHEN token RLIKE '^[0-9]+,[0-9]{2}$' THEN REPLACE(token, ',', '.')
                            WHEN token RLIKE '^[0-9]{1,3}(\\.[0-9]{3})+,[0-9]{2}$' THEN REPLACE(REPLACE(token, '.', ''), ',', '.')
                            WHEN token RLIKE '^[0-9]{1,3}(,[0-9]{3})+\\.[0-9]{2}$' THEN REPLACE(token, ',', '')
                            WHEN token RLIKE '^[0-9]+(\\.[0-9]{2})?$' THEN token
                            ELSE NULL
                        END AS DECIMAL(10,2)
                    )
                ),
                amount -> amount IS NOT NULL
            ),
            CAST(NULL AS DECIMAL(10,2)),
            (acc, amount) -> CASE WHEN acc IS NULL OR amount > acc THEN amount ELSE acc END
        ) AS Numeric_Total,
        GREATEST(COALESCE(TRY_CAST(REGEXP_REPLACE(Divisor, '[^0-9.]', '') AS DECIMAL(10,2)), 1), 1) AS Numeric_Divisor
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
)

-- Output columns:
-- Row_Num: Sequential row number.
-- AU_ID: Mapped AU from cost center bridge.
-- Account: Raw Account string from Coupa.
-- Cleaned_CC: Normalized cost center extracted from Account.
-- CatCode: Category code extracted from Account.
-- PublicOfficial: Raw PublicOfficial value.
-- Raw_Total: Original Total string from Coupa.
-- Numeric_Total: Parsed numeric amount.
-- Raw_Divisor: Original Divisor string from Coupa.
-- Numeric_Divisor: Parsed divisor (defaults to 1 if missing).
-- Per_Person_Total: Numeric_Total / Numeric_Divisor.
-- Non_PO_Pass: 1 if PublicOfficial is N/NO.
-- Threshold_Pass: 1 if Per_Person_Total > 250.
-- CatCode_Pass: 1 if CatCode is in the 29-code target list.
-- Exception_793_Pass: 1 if row passes the 793 exception rule.
-- Is_Counted: Whether this row is included in the final EBA04 count.
-- Exclusion_Reason: Why excluded, or 'Counted' if included.
SELECT
    ROW_NUMBER() OVER (ORDER BY m.AU_ID, p.Account) AS Row_Num,
    m.AU_ID,
    p.Account,
    p.Cleaned_CC,
    p.CatCode,
    p.PublicOfficial,
    p.Raw_Total,
    p.Numeric_Total,
    p.Raw_Divisor,
    p.Numeric_Divisor,
    ROUND(p.Numeric_Total / p.Numeric_Divisor, 2) AS Per_Person_Total,
    CASE WHEN UPPER(TRIM(p.PublicOfficial)) IN ('N', 'NO') THEN 1 ELSE 0 END AS Non_PO_Pass,
    CASE WHEN p.Numeric_Total IS NOT NULL AND (p.Numeric_Total / p.Numeric_Divisor) > 250 THEN 1 ELSE 0 END AS Threshold_Pass,
    CASE WHEN p.CatCode IN ('066','009','012','067','095','134','168','192','203','204','208','209','242','269','270','484','487','636','637','638','639','676','782','783','784','792','793','890','892') THEN 1 ELSE 0 END AS CatCode_Pass,
    CASE WHEN p.CatCode = '793' AND m.AU_ID != '101016' THEN 0 ELSE 1 END AS Exception_793_Pass,
    CASE
        WHEN UPPER(TRIM(p.PublicOfficial)) IN ('N', 'NO')
         AND p.Numeric_Total IS NOT NULL
         AND (p.Numeric_Total / p.Numeric_Divisor) > 250
         AND p.CatCode IN ('066','009','012','067','095','134','168','192','203','204','208','209','242','269','270','484','487','636','637','638','639','676','782','783','784','792','793','890','892')
         AND NOT (p.CatCode = '793' AND m.AU_ID != '101016')
        THEN 'Yes' ELSE 'No'
    END AS Is_Counted,
    CASE
        WHEN UPPER(TRIM(p.PublicOfficial)) NOT IN ('N', 'NO') THEN 'Public Official row'
        WHEN p.Numeric_Total IS NULL THEN 'Total could not be parsed'
        WHEN (p.Numeric_Total / p.Numeric_Divisor) <= 250 THEN CONCAT('Per-person amount $', CAST(ROUND(p.Numeric_Total / p.Numeric_Divisor, 2) AS STRING), ' <= $250')
        WHEN p.CatCode NOT IN ('066','009','012','067','095','134','168','192','203','204','208','209','242','269','270','484','487','636','637','638','639','676','782','783','784','792','793','890','892') THEN CONCAT('CatCode ', COALESCE(p.CatCode, 'NULL'), ' not in target list')
        WHEN p.CatCode = '793' AND m.AU_ID != '101016' THEN 'CatCode 793 excluded for non-101016 AU'
        ELSE 'Counted'
    END AS Exclusion_Reason
FROM Parsed_Rows p
INNER JOIN CC_Mapping m
    ON p.Cleaned_CC = m.Mapped_CC
ORDER BY m.AU_ID, p.Account